
Stacks and Queues · Study Notes

KTH  
July 31, 2026

---
Stacks support **last-in, first-out** (LIFO) semantics for inserts and deletes,
whereas queues are **first-in, first-out** (FIFO). Stacks and queues are usually
building blocks within a solution to a more complex problem — but they also make
for stand-alone problems.

These notes cover the chapter's conceptual material: stack fundamentals, the
stacks boot camp, queue and deque fundamentals, the queues boot camp, both Top
Tips tables, and the Python library knowledge for each. Code cells are runnable.

# Part 1 — Stacks

## 1. Stack fundamentals

A stack supports two basic operations — **push** and **pop**. Elements are added
(pushed) and removed (popped) in last-in, first-out order. If the stack is empty,
`pop` typically returns null or throws an exception.

**Implementation and complexity:**
- Implemented with a **linked list**, push and pop are `O(1)`.
- Implemented with an **array**, there is a maximum number of entries it can
  hold — push and pop are still `O(1)`.
- If the array is **dynamically resized**, the *amortized* time for both push
  and pop is `O(1)`.

A stack can support additional operations such as **peek**, which returns the
top of the stack without popping it.

**Figure 5.1 — Operations on a stack** (top of stack shown last):

| Step | Stack contents |
|---|---|
| (a) Initial configuration | 2, 1, 4 |
| (b) After `pop` | 2, 1 |
| (c) After `push 3` | 2, 1, 3 |

## 2. Stacks boot camp — reverse iteration

The LIFO semantics of a stack make it very useful for creating **reverse
iterators** for sequences stored in a way that makes stepping backward from a
given element difficult or impossible.

This program uses a stack to print the entries of a **singly-linked list** in
reverse order.

In [1]:
def print_linked_list_in_reverse(head):
    nodes = []
    while head:
        nodes.append(head.data)   # push each node's data as we walk forward
        head = head.next
    while nodes:
        print(nodes.pop())        # pop yields them in reverse order

In [2]:
# Minimal node type so the boot camp code is runnable
class ListNode:
    def __init__(self, data=0, next=None):
        self.data = data
        self.next = next

# Build 1 -> 2 -> 3 -> 4
head = ListNode(1, ListNode(2, ListNode(3, ListNode(4))))
print_linked_list_in_reverse(head)

4
3
2
1


**Complexity.** Both time and space are `O(n)`, where `n` is the number of
nodes in the list.

### Table 5.1 — Top Tips for Stacks
- Learn to recognize when the **stack LIFO property is applicable**. For
  example, **parsing** typically benefits from a stack.
- Consider **augmenting** the basic stack or queue data structure to support
  additional operations, such as finding the **maximum element**.

## 3. Know your stack libraries

Some problems require implementing your own stack class; for others, use the
built-in `list` type.

| Operation | Meaning |
|---|---|
| `s.append(e)` | Pushes an element onto the stack. Not much can go wrong with a push. |
| `s[-1]` | Retrieves, but does **not** remove, the element at the top. |
| `s.pop()` | Removes **and** returns the element at the top. |
| `len(s) == 0` | Tests if the stack is empty. |

⚠️ When called on an empty list `s`, both `s[-1]` and `s.pop()` raise an
**`IndexError`**.

In [3]:
s = []
s.append(2); s.append(1); s.append(4)     # push
print("stack:", s)
print("peek s[-1]:", s[-1])               # peek — does not remove
print("pop:", s.pop())                    # removes and returns 4
print("after pop:", s)
s.append(3)
print("after push 3:", s)                 # matches Figure 5.1
print("is empty:", len(s) == 0)

stack: [2, 1, 4]
peek s[-1]: 4
pop: 4
after pop: [2, 1]
after push 3: [2, 1, 3]
is empty: False


In [4]:
# Both peek and pop raise IndexError on an empty stack
empty = []
for op, fn in (("empty[-1]", lambda: empty[-1]), ("empty.pop()", empty.pop)):
    try:
        fn()
    except IndexError as e:
        print(f"{op:<12} -> IndexError: {e}")

empty[-1]    -> IndexError: list index out of range
empty.pop()  -> IndexError: pop from empty list


# Part 2 — Queues

## 4. Queue fundamentals

A queue supports two basic operations — **enqueue** and **dequeue**. (If the
queue is empty, `dequeue` typically returns null or throws an exception.)
Elements are added and removed in **first-in, first-out** order.

- The most recently inserted element is the **tail** (or **back**) element.
- The element inserted least recently is the **head** (or **front**) element.

A queue can be implemented using a **linked list**, in which case these
operations have `O(1)` time complexity. The queue API often includes other
operations, e.g., a method returning the item at the head without removing it,
or the item at the tail without removing it.

**Figure 5.3 — Enqueuing and dequeuing** (head shown first):

| Step | Queue contents |
|---|---|
| (a) Initial configuration | 3, 2, 0 |
| (b) After `dequeue` | 2, 0 |
| (c) After `enqueue 4` | 2, 0, 4 |

### Deques

A **deque** (double-ended queue) is a doubly linked list in which all insertions
and deletions happen at one of the two ends — the head or the tail. Standard
nomenclature (which varies across languages and libraries):

| Operation | End | Common name |
|---|---|---|
| Insertion at the front | head | **push** |
| Insertion at the back | tail | **inject** |
| Deletion from the front | head | **pop** |
| Deletion from the back | tail | **eject** |

## 5. Queues boot camp — a queue with a max API

This implements the basic queue API — `enqueue` and `dequeue` — plus a `max`
method returning the maximum element in the queue. The core idea is
**composition**: hold a private field referencing a library queue object and
forward the existing methods to it.

In [ ]:
class Queue:
    def __init__(self):
        self._data = []

    def enqueue(self, x):
        self._data.append(x)

    def dequeue(self):
        return self._data.pop(0)

    def max(self):
        return max(self._data)

In [ ]:
q = Queue()
for x in (3, 2, 0):
    q.enqueue(x)
print("max:", q.max())
print("dequeue:", q.dequeue())    # FIFO -> 3
q.enqueue(4)
print("max after enqueue 4:", q.max())

**Complexity.** `enqueue` and `dequeue` have the same complexity as the
underlying library queue, namely `O(1)`. Finding the maximum is `O(n)`, where
`n` is the number of entries.

> 📝 **Note on the boot camp code.** It backs the queue with a `list` and
> dequeues via `_data.pop(0)`, which is actually `O(n)` for a Python list —
> every remaining element shifts left. The `O(1)` claim holds for a proper queue
> implementation; `collections.deque` (below) gives genuine `O(1)` at both ends.

### Table 5.2 — Top Tips for Queues
- Learn to recognize when the **queue FIFO property is applicable**. For
  example, queues are ideal when **order needs to be preserved**.

## 6. Know your queue libraries

Some problems require implementing your own queue class; for others, use
**`collections.deque`**.

| Operation | Meaning |
|---|---|
| `q.append(e)` | Pushes an element onto the queue. Not much can go wrong. |
| `q[0]` | Retrieves, but does not remove, the element at the **front**. |
| `q[-1]` | Retrieves, but does not remove, the element at the **back**. |
| `q.popleft()` | Removes **and** returns the element at the front. |

⚠️ Dequeuing from, or accessing the head/tail of, an **empty** collection raises
an **`IndexError`**.

In [5]:
import collections

q = collections.deque()
for x in (3, 2, 0):
    q.append(x)
print("queue:", q)
print("front q[0]:", q[0], "| back q[-1]:", q[-1])
print("popleft:", q.popleft())          # FIFO -> 3
q.append(4)
print("after enqueue 4:", q)            # matches Figure 5.3

queue: deque([3, 2, 0])
front q[0]: 3 | back q[-1]: 0
popleft: 3
after enqueue 4: deque([2, 0, 4])


In [6]:
# deque also supports the full double-ended (deque) vocabulary
d = collections.deque([1, 2, 3])
d.appendleft(0)    # "push"   — insert at front
d.append(4)        # "inject" — insert at back
print("after appendleft/append:", d)
print("popleft (pop):", d.popleft(), "| pop (eject):", d.pop())
print("final:", d)

# IndexError on an empty deque
empty = collections.deque()
try:
    empty.popleft()
except IndexError as e:
    print("empty.popleft() -> IndexError:", e)

after appendleft/append: deque([0, 1, 2, 3, 4])
popleft (pop): 0 | pop (eject): 4
final: deque([1, 2, 3])
empty.popleft() -> IndexError: pop from an empty deque
